In [ ]:
#@title ARC-AGI-2 P141 TorchAO-Compatible Dual-Seed LOPO Lab { display-mode: "form" }
#@markdown ### 1. Identidade do experimento
EXPERIMENT_ID = "HYP-P141-torchao-017-torch211-contract" #@param {type:"string"}
EXPERIMENT_NOTE = "P140 corrigida: TorchAO 0.17 ABI-estavel com Torch 2.11 e contrato Unsloth completo" #@param {type:"string"}
RUN_ID_SUFFIX = "" #@param {type:"string"}
#@markdown ### 1b. Protocolo supervisionado sem leakage
PILOT_TASKS = 16 #@param {type:"integer"}
EPISODES_PER_TASK = 1 #@param {type:"integer"}
OUTER_FOLDS = 5 #@param {type:"integer"}
AUTOLEARN_SEED = 20260722 #@param {type:"integer"}
BOOTSTRAP_SAMPLES = 2000 #@param {type:"integer"}
ENABLE_FULL_PROCESS_TRACE = True #@param {type:"boolean"}
STOP_ON_AUTOLEARN_FAILURE = True #@param {type:"boolean"}
#@markdown ### 2. Bundle e subset
BUNDLE_NAME = 'arc_agi2_autolearning_bundle_p141_20260723.zip' #@param {type:"string"}
TRY_DRIVE_MOUNT = True #@param {type:"boolean"}
RUN_KEYS = "0934a4d8,36a08778,981571dc,aa4ec2a5,e8686506,7666fa5d,135a2760,80a900e0,2c181942,9aaea919,20270e3b,9385bd28,269e22fb,4c7dc4dd,d8e07eb2,d35bdbdc" #@param {type:"string"}
MAX_TASKS = 16 #@param {type:"integer"}
SECONDS_PER_PROFILE_MINUTES = 420 #@param {type:"integer"}
#@markdown ### 3. Perfis Qwen
PROFILE_PRESET = "canonical_only" #@param ["canonical_only", "baseline_only", "baseline_plus_diverse", "baseline_plus_diverse_deep", "baseline_plus_deep", "custom"]
CUSTOM_PROFILES = "koushik_plus,koushik_diverse,koushik_deep" #@param {type:"string"}
#@markdown ### 3b. Matriz de geração. O preset recomendado altera somente as sementes.
PORTFOLIO_PRESET = "dual_seed_koushik" #@param ["dual_seed_koushik", "off", "custom"]
CUSTOM_RUN_MATRIX_JSON = "[]" #@param {type:"string"}
#@markdown ### 4. Seletor e gates
SELECTOR_PRESET = "kgmon" #@param ["kgmon", "submit_public_3389", "topology_second", "portfolio", "custom"]
CUSTOM_SELECTOR_WEIGHTS = "selection_mode=public_kgmon" #@param {type:"string"}
MAX_DUPLICATE_ATTEMPT_RATE = 0.15 #@param {type:"number"}
MAX_ATTEMPT2_INPUT_FALLBACK_RATE = 0.15 #@param {type:"number"}
USE_SYMBOLIC = False #@param {type:"boolean"}
MISSING_SYMBOLIC_FALLBACK = True #@param {type:"boolean"}
STOP_AFTER_BASELINE_FAILURE = True #@param {type:"boolean"}
#@markdown ### 4b. Sweep barato de selector, sem refazer inferencia
SELECTOR_SWEEP_ENABLED = True #@param {type:"boolean"}
SELECTOR_SWEEP_MODES = "public_3389,public_3389_topology_second,public_3389_portfolio_first,public_3389_vote_first,public_probmul,public_kgmon,public_portfolio,portfolio" #@param {type:"string"}
#@markdown ### 5. Overrides avancados do perfil Qwen. Deixe vazio para usar o perfil padrao.
TRAIN_AUG_N = "" #@param {type:"string"}
EVAL_AUG_N = "" #@param {type:"string"}
DFS_SECONDS = "" #@param {type:"string"}
PUZZLE_TIMEOUT_SECONDS = "" #@param {type:"string"}
MIN_START_REMAINING_SECONDS = "" #@param {type:"string"}
MAX_SCORE_PROB = "" #@param {type:"string"}
TRAIN_PRECISION = "auto" #@param ["auto", "bf16", "fp16", "fp32"]
#@markdown ### 6. Runtime e staging
FORCE_GPU_COUNT = "1" #@param ["1", "2", "4"] {allow-input: true}
REQUIRE_L4_TIMING = False #@param {type:"boolean"}
INSTALL_COMPAT_UNSLOTH = "auto" #@param ["auto", "force", "skip"]
STRICT_FLASH_CAUSAL = False #@param {type:"boolean"}
QWEN3_PATCH_OVERLAY_MODE = "0" #@param ["0", "1", "force"]
#@markdown ### 7. Logs
HF_LOG_ENABLED_FORM = True #@param {type:"boolean"}
HF_LOG_DATASET_FORM = "" #@param {type:"string"}
HF_LOG_SYNC_SECONDS_FORM = 180 #@param {type:"integer"}
DRIVE_LOG_ROOT_FORM = "/content/drive/MyDrive/arc2016_colab_live_logs" #@param {type:"string"}
DRIVE_LOG_SYNC_SECONDS_FORM = 30 #@param {type:"integer"}
#@markdown ---
#@markdown **Regra:** este notebook coleta evidencia e nunca submete ao Kaggle.

import json
import base64
import hashlib
import logging
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import shlex
import subprocess
import sys
import threading
import time
import traceback
import warnings
import zipfile


ROOT_DIR = "/content/arc_agi2_autolearning_p141"
LAB_CONFIG_VERSION = "p141-torchao017-torch211-runtime-hardened-lopo"
EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_BUNDLE_SHA256 = 'e5d8244d4b6b3f5df91496b27606b5879b786b799fdad57d62e91ba24ad4737e'
COLAB_RELEASE_POLICY = (
    "Never update an opened Colab notebook file in place. "
    "Each release gets a new Drive file ID and a versioned bundle."
)
P141_ATOMIC_SPECS = [
    "kaggle==2.2.3",
    "unsloth==2025.9.7",
    "unsloth_zoo==2025.9.9",
    "transformers==4.55.4",
    "peft==0.18.1",
    "datasets==3.6.0",
    "accelerate==1.13.0",
    "trl==0.22.2",
    "bitsandbytes==0.48.2",
    "xformers==0.0.35",
    "huggingface_hub==0.36.2",
    "tokenizers==0.21.4",
    "torchao==0.17.0",
    "scikit-learn==1.7.2",
]
COLAB_COMPAT_UNSLOTH_SPEC = " ".join(P141_ATOMIC_SPECS)
P141_EXPECTED_EXACT = {
    "kaggle": "2.2.3",
    "unsloth": "2025.9.7",
    "unsloth_zoo": "2025.9.9",
    "transformers": "4.55.4",
    "peft": "0.18.1",
    "datasets": "3.6.0",
    "accelerate": "1.13.0",
    "trl": "0.22.2",
    "bitsandbytes": "0.48.2",
    "xformers": "0.0.35",
    "huggingface-hub": "0.36.2",
    "tokenizers": "0.21.4",
    "torchao": "0.17.0",
    "scikit-learn": "1.7.2",
}
P141_EXPECTED_RUNTIME_PUBLIC = {
    "torch": "2.11.0",
    "torchvision": "0.26.0",
    "triton": "3.6.0",
}
DEPENDENCY_CONTRACT_PATH = Path(ROOT_DIR) / "dependency_contract_p141.json"
FLASH_CAUSAL_STRICT = os.environ.get(
    "ARC_COLAB_STRICT_FLASH_CAUSAL",
    "1" if bool(STRICT_FLASH_CAUSAL) else "0",
).strip()
QWEN3_PATCH_OVERLAY = os.environ.get(
    "ARC_COLAB_QWEN3_PATCH_OVERLAY",
    str(QWEN3_PATCH_OVERLAY_MODE),
).strip().lower()


def csv_items(text):
    return [part.strip() for part in str(text or "").split(",") if part.strip()]


if Path(BUNDLE_NAME).name != BUNDLE_NAME or not str(BUNDLE_NAME).endswith(".zip"):
    raise ValueError("BUNDLE_NAME must be a plain .zip filename, not a path")

RUN_ID_SUFFIX = re.sub(r"[^0-9A-Za-z_.-]+", "-", str(RUN_ID_SUFFIX or "").strip()).strip("-_.")[:60]

RUN_KEYS_LIST = csv_items(RUN_KEYS)
if not RUN_KEYS_LIST:
    raise ValueError("RUN_KEYS cannot be empty")
invalid_run_keys = [key for key in RUN_KEYS_LIST if not re.fullmatch(r"[0-9a-fA-F]{8}", key)]
if invalid_run_keys:
    raise ValueError(f"RUN_KEYS has invalid ARC task ids: {invalid_run_keys}")
if len(RUN_KEYS_LIST) != len(set(RUN_KEYS_LIST)):
    raise ValueError("RUN_KEYS must not contain duplicates")
RUN_KEYS = ",".join(RUN_KEYS_LIST)
CONFIGURED_RUN_KEYS_LIST = tuple(RUN_KEYS_LIST)
CONFIGURED_RUN_KEYS = RUN_KEYS
EFFECTIVE_LOPO_KEYS_LIST = ()
EFFECTIVE_LOPO_KEYS = ""

MAX_TASKS = int(MAX_TASKS)
if not 1 <= MAX_TASKS <= 16:
    raise ValueError("MAX_TASKS must be between 1 and 16 for this lab notebook")

SECONDS_PER_PROFILE_MINUTES = int(SECONDS_PER_PROFILE_MINUTES)
if not 5 <= SECONDS_PER_PROFILE_MINUTES <= 600:
    raise ValueError("SECONDS_PER_PROFILE_MINUTES must be between 5 and 600")

PROFILE_PRESETS = {
    "canonical_only": ["koushik"],
    "baseline_plus_diverse_deep": ["koushik_plus", "koushik_diverse", "koushik_deep"],
    "baseline_only": ["koushik_plus"],
    "baseline_plus_deep": ["koushik_plus", "koushik_deep"],
    "baseline_plus_diverse": ["koushik_plus", "koushik_diverse"],
}
PROFILES = csv_items(CUSTOM_PROFILES) if PROFILE_PRESET == "custom" else PROFILE_PRESETS.get(PROFILE_PRESET, [])
if not PROFILES or PROFILES[0] not in {"koushik", "koushik_plus"}:
    raise ValueError("PROFILES must start with baseline profile 'koushik' or 'koushik_plus'")
if any(not re.fullmatch(r"[0-9A-Za-z_.-]+", profile) for profile in PROFILES):
    raise ValueError(f"PROFILES contains unsafe profile names: {PROFILES}")
if len(PROFILES) != len(set(PROFILES)):
    raise ValueError("PROFILES must not contain duplicates")

DUAL_SEED_RUN_MATRIX = [
    {
        "tag": "seed-a",
        "profile": "koushik",
        "lora_rank": 256,
        "train_aug_n": 16,
        "eval_aug_n": 2,
        "dfs_seconds": 540,
        "puzzle_timeout_seconds": 1200,
        "max_score_prob": 0.2,
        "global_seed": 42,
        "peft_random_state": 42,
        "train_seed": 42,
        "train_aug_seed": 1,
        "eval_aug_seed": 2,
        "puzzle_seed_salt": "",
        "score_aug_seed_salt": "",
    },
    {
        "tag": "seed-b",
        "profile": "koushik",
        "lora_rank": 256,
        "train_aug_n": 16,
        "eval_aug_n": 2,
        "dfs_seconds": 540,
        "puzzle_timeout_seconds": 1200,
        "max_score_prob": 0.2,
        "global_seed": 314159,
        "peft_random_state": 271828,
        "train_seed": 161803,
        "train_aug_seed": 104729,
        "eval_aug_seed": 130363,
        "puzzle_seed_salt": "dual-b-puzzle",
        "score_aug_seed_salt": "dual-b-score",
    },
]
PORTFOLIO_PRESET = str(PORTFOLIO_PRESET).strip().lower()
if PORTFOLIO_PRESET == "dual_seed_koushik":
    RUN_MATRIX = DUAL_SEED_RUN_MATRIX
elif PORTFOLIO_PRESET == "off":
    RUN_MATRIX = []
elif PORTFOLIO_PRESET == "custom":
    try:
        RUN_MATRIX = json.loads(str(CUSTOM_RUN_MATRIX_JSON or "[]"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"CUSTOM_RUN_MATRIX_JSON is invalid JSON: {exc.msg}") from exc
else:
    raise ValueError("PORTFOLIO_PRESET must be dual_seed_koushik, off, or custom")
if not isinstance(RUN_MATRIX, list):
    raise ValueError("RUN_MATRIX must be a JSON list")
if RUN_MATRIX and len(PROFILES) != 1:
    raise ValueError("Portfolio mode requires exactly one outer profile; choose a one-profile preset")

SELECTOR_PRESETS = {
    "kgmon": "selection_mode=public_kgmon",
    "topology_second": "selection_mode=public_3389_topology_second",
    "submit_public_3389": "selection_mode=public_3389",
    "portfolio": "selection_mode=portfolio",
}
if SELECTOR_PRESET == "custom":
    SELECTOR_WEIGHT_SPEC = CUSTOM_SELECTOR_WEIGHTS.strip()
elif SELECTOR_PRESET in SELECTOR_PRESETS:
    SELECTOR_WEIGHT_SPEC = SELECTOR_PRESETS[SELECTOR_PRESET]
else:
    raise ValueError(f"Unknown SELECTOR_PRESET={SELECTOR_PRESET!r}")
if not SELECTOR_WEIGHT_SPEC:
    raise ValueError("SELECTOR_WEIGHT_SPEC cannot be empty")
SELECTOR_SWEEP_MODES = ",".join(csv_items(SELECTOR_SWEEP_MODES))
if SELECTOR_SWEEP_ENABLED and not SELECTOR_SWEEP_MODES:
    raise ValueError("SELECTOR_SWEEP_MODES cannot be empty when SELECTOR_SWEEP_ENABLED is true")

SECONDS_PER_PROFILE = SECONDS_PER_PROFILE_MINUTES * 60
if not 0.0 <= float(MAX_DUPLICATE_ATTEMPT_RATE) <= 1.0:
    raise ValueError("MAX_DUPLICATE_ATTEMPT_RATE must be between 0 and 1")
if not 0.0 <= float(MAX_ATTEMPT2_INPUT_FALLBACK_RATE) <= 1.0:
    raise ValueError("MAX_ATTEMPT2_INPUT_FALLBACK_RATE must be between 0 and 1")
PORTFOLIO_RUN_COUNT = len(RUN_MATRIX) if RUN_MATRIX else 1
NOMINAL_SECONDS_PER_PORTFOLIO_RUN = SECONDS_PER_PROFILE / PORTFOLIO_RUN_COUNT
FROZEN_P137_REFERENCE = {
    "source": "P137 run arc2016-colab-qwen-ab-20260722T220700Z",
    "evidence_scope": "public_training_lopo_proxy_not_kaggle_score",
    "profile": "koushik",
    "portfolio_preset": "off",
    "selector_score": 0.8125,
    "oracle_score": 0.8125,
    "selector_correct_outputs": 13,
    "oracle_correct_outputs": 13,
    "outputs_total": 16,
    "returncode": 0,
    "total_budget_seconds": 25200,
}
if SECONDS_PER_PROFILE != FROZEN_P137_REFERENCE["total_budget_seconds"]:
    raise ValueError("P141 must preserve the P137 total generator budget for a comparable portfolio-policy test")
FORCE_GPU_COUNT = str(FORCE_GPU_COUNT).strip()
if FORCE_GPU_COUNT not in {"1", "2", "4"}:
    raise ValueError("FORCE_GPU_COUNT must be one of 1, 2, 4")
INSTALL_COMPAT_UNSLOTH = str(INSTALL_COMPAT_UNSLOTH).strip().lower()
if INSTALL_COMPAT_UNSLOTH not in {"auto", "force", "skip"}:
    raise ValueError("INSTALL_COMPAT_UNSLOTH must be auto, force, or skip")
HF_LOG_SYNC_SECONDS = int(HF_LOG_SYNC_SECONDS_FORM)
if not 15 <= HF_LOG_SYNC_SECONDS <= 600:
    raise ValueError("HF_LOG_SYNC_SECONDS_FORM must be between 15 and 600")
DRIVE_LOG_SYNC_SECONDS = int(DRIVE_LOG_SYNC_SECONDS_FORM)
if not 10 <= DRIVE_LOG_SYNC_SECONDS <= 600:
    raise ValueError("DRIVE_LOG_SYNC_SECONDS_FORM must be between 10 and 600")
DRIVE_LOG_ROOT = str(DRIVE_LOG_ROOT_FORM or "").strip() or "/content/drive/MyDrive/arc2016_colab_live_logs"

QWEN_OPTIONAL_OVERRIDES = {
    "ARC_QWEN_TRAIN_AUG_N": TRAIN_AUG_N,
    "ARC_QWEN_EVAL_AUG_N": EVAL_AUG_N,
    "ARC_QWEN_DFS_SECONDS": DFS_SECONDS,
    "ARC_QWEN_PUZZLE_TIMEOUT_SECONDS": PUZZLE_TIMEOUT_SECONDS,
    "ARC_QWEN_MIN_START_REMAINING_SECONDS": MIN_START_REMAINING_SECONDS,
    "ARC_QWEN_MAX_SCORE_PROB": MAX_SCORE_PROB,
    "ARC_QWEN_TRAIN_PRECISION": TRAIN_PRECISION,
}
QWEN_OPTIONAL_OVERRIDES = {
    key: str(value).strip()
    for key, value in QWEN_OPTIONAL_OVERRIDES.items()
    if str(value).strip()
}
if RUN_MATRIX:
    QWEN_OPTIONAL_OVERRIDES["ARC_QWEN_RUN_MATRIX_JSON"] = json.dumps(
        RUN_MATRIX, separators=(",", ":"), sort_keys=True
    )
    QWEN_OPTIONAL_OVERRIDES["ARC_QWEN_PORTFOLIO_CONTINUE_ON_ERROR"] = "0"

# Keep Kaggle kernel-output staging bounded. This mirrors
# hf_stage_kaggle_assets.DEFAULT_KAGGLE_OUTPUT_PATTERN and also protects reruns
# that accidentally use an older bundle where the default was not applied.
KAGGLE_OUTPUT_FILE_PATTERN = (
    r"^(unsloth|unsloth_zoo|trl|bitsandbytes|flash_attn|cut_cross_entropy|"
    r"xformers|triton|tyro|shtab|docstring_parser)(/|-)"
)

# HF logging. Leave ARC_HF_LOG_DATASET empty to auto-create/use
# <hf-username>/arc-2016-colab-logs as a private dataset.
HF_LOG_ENABLED = bool(HF_LOG_ENABLED_FORM) and os.environ.get("ARC_HF_LOG_ENABLED", "1").lower() not in {"0", "false", "no"}
HF_LOG_DATASET = os.environ.get("ARC_HF_LOG_DATASET") or str(HF_LOG_DATASET_FORM or "").strip()
RUN_ID_BASE = f"arc2016-colab-qwen-ab-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}-{time.time_ns() % 1_000_000_000:09d}"
RUN_ID = os.environ.get("ARC_COLAB_RUN_ID") or (f"{RUN_ID_BASE}-{RUN_ID_SUFFIX}" if RUN_ID_SUFFIX else RUN_ID_BASE)
HF_BRIDGE = None
DRIVE_LOG_MIRROR = None

# Keep logs focused on actionable events. These filters only silence known,
# non-critical notebook/HF noise; exceptions and command failures still surface.
QUIET_ENV_DEFAULTS = {
    "TF_CPP_MIN_LOG_LEVEL": "3",
    "TF_ENABLE_ONEDNN_OPTS": "0",
    "USE_TF": "0",
    "USE_FLAX": "0",
    "TOKENIZERS_PARALLELISM": "false",
}
for key, value in QUIET_ENV_DEFAULTS.items():
    os.environ.setdefault(key, value)

NONCRITICAL_LOG_PATTERNS = [
    re.compile(r"WARNING: unsloth .* does not provide the extra 'triton'"),
    re.compile(r".*tensorflow/core/util/port\.cc:.*oneDNN custom operations are on.*"),
    re.compile(r".*tensorflow/core/platform/cpu_feature_guard\.cc:.*optimized to use available CPU instructions.*"),
    re.compile(r"To enable the following instructions: .*"),
    re.compile(r"Flax classes are deprecated and will be removed in Diffusers.*"),
    re.compile(r".*UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4\.56\.0\..*"),
    re.compile(r"\s*_install_fused_forward\(\)\s*$"),
]
warnings.filterwarnings("ignore", message=r".*No files have been modified since last commit.*")
warnings.filterwarnings("ignore", message=r".*resume_download.*deprecated.*")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)


def is_noncritical_log_line(line: str) -> bool:
    return any(pattern.match(line.rstrip()) for pattern in NONCRITICAL_LOG_PATTERNS)


def section(title: str) -> None:
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("section", {"title": title})


LAB_PARAMETERS = {
    "lab_config_version": LAB_CONFIG_VERSION,
    "experiment_id": EXPERIMENT_ID,
    "experiment_note": EXPERIMENT_NOTE,
    "run_id_suffix": RUN_ID_SUFFIX,
    "run_id": RUN_ID,
    "bundle_name": BUNDLE_NAME,
    "embedded_bundle_sha256": EMBEDDED_BUNDLE_SHA256,
    "try_drive_mount": bool(TRY_DRIVE_MOUNT),
    "configured_run_keys": CONFIGURED_RUN_KEYS,
    "configured_run_key_count": len(CONFIGURED_RUN_KEYS_LIST),
    "effective_lopo_keys": EFFECTIVE_LOPO_KEYS,
    "effective_lopo_key_count": len(EFFECTIVE_LOPO_KEYS_LIST),
    "max_tasks": int(MAX_TASKS),
    "seconds_per_profile": int(SECONDS_PER_PROFILE),
    "profiles": PROFILES,
    "profile_preset": PROFILE_PRESET,
    "portfolio_preset": PORTFOLIO_PRESET,
    "run_matrix": RUN_MATRIX,
    "portfolio_run_count": PORTFOLIO_RUN_COUNT,
    "nominal_seconds_per_portfolio_run": NOMINAL_SECONDS_PER_PORTFOLIO_RUN,
    "frozen_p137_reference": FROZEN_P137_REFERENCE,
    "selector_preset": SELECTOR_PRESET,
    "selector_weight_spec": SELECTOR_WEIGHT_SPEC,
    "selector_sweep_enabled": bool(SELECTOR_SWEEP_ENABLED),
    "selector_sweep_modes": SELECTOR_SWEEP_MODES,
    "max_duplicate_attempt_rate": float(MAX_DUPLICATE_ATTEMPT_RATE),
    "max_attempt2_input_fallback_rate": float(MAX_ATTEMPT2_INPUT_FALLBACK_RATE),
    "use_symbolic": bool(USE_SYMBOLIC),
    "missing_symbolic_fallback": bool(MISSING_SYMBOLIC_FALLBACK),
    "stop_after_baseline_failure": bool(STOP_AFTER_BASELINE_FAILURE),
    "qwen_optional_overrides": QWEN_OPTIONAL_OVERRIDES,
    "force_gpu_count": str(FORCE_GPU_COUNT),
    "require_l4_timing": bool(REQUIRE_L4_TIMING),
    "strict_flash_causal": FLASH_CAUSAL_STRICT,
    "qwen3_patch_overlay": QWEN3_PATCH_OVERLAY,
    "install_compat_unsloth": INSTALL_COMPAT_UNSLOTH,
    "hf_log_enabled": bool(HF_LOG_ENABLED),
    "hf_log_dataset": HF_LOG_DATASET,
    "hf_log_sync_seconds": int(HF_LOG_SYNC_SECONDS),
    "drive_log_root": DRIVE_LOG_ROOT,
    "drive_log_sync_seconds": int(DRIVE_LOG_SYNC_SECONDS),
}


def runtime_resource_snapshot() -> dict:
    snapshot = {}
    try:
        usage = shutil.disk_usage("/content")
        snapshot["disk_free_bytes"] = usage.free
    except Exception as exc:
        snapshot["disk_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    try:
        probe = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
            text=True, capture_output=True, check=False, timeout=5,
        )
        snapshot["nvidia_smi_returncode"] = probe.returncode
        snapshot["gpu_resource_rows"] = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
    except Exception as exc:
        snapshot["gpu_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    return snapshot


class TeeStream:
    def __init__(self, stream, path: Path):
        self.stream = stream
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(path, "a", encoding="utf-8", buffering=1)
        self._lock = threading.RLock()

    def write(self, data):
        with self._lock:
            self.stream.write(data)
            self.file.write(data)
        return len(data)

    def flush(self):
        with self._lock:
            self.stream.flush()
            self.file.flush()

    def close(self):
        with self._lock:
            if not self.file.closed:
                self.file.flush()
                self.file.close()

    def __getattr__(self, name):
        return getattr(self.stream, name)


class HFLogBridge:
    def __init__(self, *, token: str, run_id: str, dataset_repo: str, sync_seconds: int):
        self.token = token
        self.run_id = run_id
        self.sync_seconds = max(15, int(sync_seconds))
        self.log_dir = Path("/content/arc2016_hf_logs") / run_id
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.stdout_path = self.log_dir / "stdout.log"
        self.stderr_path = self.log_dir / "stderr.log"
        self.events_path = self.log_dir / "events.jsonl"
        self.heartbeat_path = self.log_dir / "heartbeat.json"
        self.summary_path = self.log_dir / "run_summary.json"
        self.artifact_index_path = self.log_dir / "artifact_upload_index.json"
        self.enabled = False
        self.repo_id = dataset_repo
        self.repo_url = None
        self.api = None
        self._stop = threading.Event()
        self._lock = threading.RLock()
        self._sync_lock = threading.Lock()
        self._thread = None
        self._sync_errors = 0
        self._uploaded_signatures = {}
        self._runtime_state = {
            "active_phase": "initialization",
            "active_command": None,
            "last_progress_utc": None,
            "command_started_epoch": None,
        }

        if not HF_LOG_ENABLED:
            self.event("hf_logging_disabled", {"reason": "ARC_HF_LOG_ENABLED=0"}, upload=False)
            return
        if not token:
            self.event("hf_logging_disabled", {"reason": "missing HF_TOKEN/HF_KEY"}, upload=False)
            return

        try:
            try:
                from huggingface_hub import HfApi
            except Exception:
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34.0,<1.0"],
                    check=True,
                )
                from huggingface_hub import HfApi

            self.api = HfApi(token=token)
            who = self.api.whoami(token=token)
            username = who.get("name") or who.get("fullname") or who.get("email", "unknown").split("@")[0]
            if not self.repo_id:
                self.repo_id = f"{username}/arc-2016-colab-logs"
            self.api.create_repo(
                repo_id=self.repo_id,
                repo_type="dataset",
                private=True,
                exist_ok=True,
                token=token,
            )
            self.repo_url = f"https://huggingface.co/datasets/{self.repo_id}/tree/main/runs/{self.run_id}"
            self.enabled = True
            self.event("hf_logging_started", {
                "repo_id": self.repo_id,
                "repo_url": self.repo_url,
                "sync_seconds": self.sync_seconds,
            }, upload=False)
            self.write_heartbeat("started")
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        except Exception as exc:
            self.enabled = False
            self.event("hf_logging_start_failed", {"error": repr(exc)}, upload=False)

    def event(self, name: str, payload: dict | None = None, *, upload: bool = False) -> None:
        record = {
            "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "run_id": self.run_id,
            "event": name,
            "payload": payload or {},
        }
        with self._lock:
            with open(self.events_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=True, sort_keys=True) + "\n")
        if upload:
            self.sync_once(heartbeat_status=None)

    def update_runtime_state(self, **values) -> None:
        with self._lock:
            self._runtime_state.update(values)

    def write_heartbeat(self, status: str, extra: dict | None = None) -> None:
        data = {
            "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "run_id": self.run_id,
            "status": status,
            "repo_id": self.repo_id,
            "repo_url": self.repo_url,
            "sync_errors": self._sync_errors,
        }
        data.update(self._runtime_state)
        data.update(runtime_resource_snapshot())
        if extra:
            data.update(extra)
        self.heartbeat_path.write_text(json.dumps(data, ensure_ascii=True, indent=2), encoding="utf-8")

    def write_summary(self, status: str, extra: dict | None = None) -> None:
        data = {
            "run_id": self.run_id,
            "status": status,
            "repo_id": self.repo_id,
            "repo_url": self.repo_url,
            "lab_config_version": LAB_CONFIG_VERSION,
            "experiment_id": EXPERIMENT_ID,
            "experiment_note": EXPERIMENT_NOTE,
            "bundle_name": BUNDLE_NAME,
            "profiles": PROFILES,
            "configured_run_keys": CONFIGURED_RUN_KEYS,
            "effective_lopo_keys": EFFECTIVE_LOPO_KEYS,
            "max_tasks": MAX_TASKS,
            "seconds_per_profile": SECONDS_PER_PROFILE,
            "force_gpu_count": FORCE_GPU_COUNT,
            "selector_weight_spec": SELECTOR_WEIGHT_SPEC,
            "lab_parameters": LAB_PARAMETERS,
        }
        if extra:
            data.update(extra)
        self.summary_path.write_text(json.dumps(data, ensure_ascii=True, indent=2), encoding="utf-8")

    @staticmethod
    def _sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with open(path, "rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def _upload_file(self, path: Path, path_in_repo: str | None = None) -> str:
        if not self.enabled or self.api is None:
            return "disabled"
        if not path.exists() or not path.is_file():
            return "missing"
        path_in_repo = path_in_repo or f"runs/{self.run_id}/{path.name}"
        stat = path.stat()
        signature = (stat.st_size, stat.st_mtime_ns)
        if self._uploaded_signatures.get(path_in_repo) == signature:
            return "unchanged"
        for attempt in range(1, 5):
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message=r".*No files have been modified since last commit.*")
                    self.api.upload_file(
                        path_or_fileobj=str(path),
                        path_in_repo=path_in_repo,
                        repo_id=self.repo_id,
                        repo_type="dataset",
                        token=self.token,
                    )
                self._uploaded_signatures[path_in_repo] = signature
                return "uploaded"
            except Exception as exc:
                rate_limited = "429" in repr(exc) or "Too Many Requests" in repr(exc)
                if attempt < 4 and rate_limited:
                    time.sleep(min(60, 2 ** attempt * 5))
                    continue
                raise
        return "failed"

    def sync_once(
        self,
        extra_paths: list[Path] | None = None,
        heartbeat_status: str | None = "running",
        *,
        wait_for_lock: bool = False,
    ) -> bool:
        if not self.enabled:
            return False
        acquired = (
            self._sync_lock.acquire(timeout=60)
            if wait_for_lock
            else self._sync_lock.acquire(blocking=False)
        )
        if not acquired:
            return False
        try:
            self._sync_once_impl(extra_paths=extra_paths, heartbeat_status=heartbeat_status)
        finally:
            self._sync_lock.release()
        return True

    def _sync_once_impl(self, extra_paths: list[Path] | None = None, heartbeat_status: str | None = "running") -> None:
        if not self.enabled:
            return
        if heartbeat_status is not None:
            self.write_heartbeat(heartbeat_status)
        records = []
        targets = [
            (path, f"runs/{self.run_id}/{path.name}")
            for path in [self.events_path, self.stdout_path, self.stderr_path, self.heartbeat_path, self.summary_path]
        ]
        seen_remote = {remote for _, remote in targets}
        for value in extra_paths or []:
            path = Path(value)
            path_identity = hashlib.sha256(str(path.resolve()).encode("utf-8")).hexdigest()[:12]
            remote = f"runs/{self.run_id}/artifacts/{path_identity}_{path.name}"
            if remote in seen_remote:
                records.append({"local_path": str(path), "remote_path": remote, "status": "duplicate_remote_skipped"})
                continue
            seen_remote.add(remote)
            targets.append((path, remote))
        for path, remote in targets:
            record = {"local_path": str(path), "remote_path": remote}
            try:
                record["status"] = self._upload_file(path, remote)
                if path.exists() and path.is_file():
                    record["size_bytes"] = path.stat().st_size
                    record["sha256"] = self._sha256(path)
            except Exception as exc:
                self._sync_errors += 1
                record["status"] = "error"
                record["error"] = f"{type(exc).__name__}: {exc}"[:500]
                self.event("hf_sync_file_error", {**record, "count": self._sync_errors}, upload=False)
            records.append(record)
        index = {
            "schema_version": 1,
            "run_id": self.run_id,
            "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "sync_errors": self._sync_errors,
            "records": records,
        }
        self.artifact_index_path.write_text(json.dumps(index, ensure_ascii=True, indent=2), encoding="utf-8")
        try:
            self._upload_file(self.artifact_index_path)
            self._upload_file(self.events_path)
        except Exception as exc:
            self._sync_errors += 1
            self.event("hf_sync_index_error", {"error": repr(exc), "count": self._sync_errors}, upload=False)

    def _loop(self) -> None:
        while not self._stop.wait(self.sync_seconds):
            try:
                self.sync_once()
            except Exception as exc:
                self._sync_errors += 1
                self.event("hf_sync_loop_error", {"error": repr(exc), "count": self._sync_errors}, upload=False)

    def stop(self, status: str = "stopped", extra_paths: list[Path] | None = None, extra: dict | None = None) -> None:
        self._stop.set()
        if self._thread is not None and self._thread is not threading.current_thread():
            self._thread.join(timeout=min(30, self.sync_seconds))
        self.update_runtime_state(active_phase=status, active_command=None, command_started_epoch=None)
        self.write_summary(status, extra=extra)
        self.event(f"hf_logging_{status}", extra or {}, upload=False)
        self.write_heartbeat(status, extra=extra)
        sys.stdout.flush()
        sys.stderr.flush()
        final_sync_ok = self.sync_once(
            extra_paths=extra_paths,
            heartbeat_status=None,
            wait_for_lock=True,
        )
        if not final_sync_ok:
            self.event("hf_final_sync_lock_timeout", {"status": status}, upload=False)
            self.write_heartbeat(status, extra={**(extra or {}), "final_sync_ok": False})


class DriveLogMirror:
    def __init__(self, source_dir: Path, dest_dir: Path, sync_seconds: int):
        self.source_dir = Path(source_dir)
        self.dest_dir = Path(dest_dir)
        self.sync_seconds = max(10, int(sync_seconds))
        self.dest_dir.mkdir(parents=True, exist_ok=True)
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.Lock()
        self._copied_signatures = {}
        self._sync_errors = 0

    def _copy_file(self, path: Path) -> None:
        if not path.exists() or not path.is_file():
            return
        rel = path.relative_to(self.source_dir)
        dest = self.dest_dir / rel
        stat = path.stat()
        signature = (stat.st_size, stat.st_mtime_ns)
        key = rel.as_posix()
        if self._copied_signatures.get(key) == signature:
            return
        dest.parent.mkdir(parents=True, exist_ok=True)
        temp = dest.with_name(dest.name + f".tmp.{os.getpid()}")
        shutil.copy2(path, temp)
        os.replace(temp, dest)
        self._copied_signatures[key] = signature

    def sync_once(self, *, wait_for_lock: bool = False) -> bool:
        acquired = self._lock.acquire(timeout=60) if wait_for_lock else self._lock.acquire(blocking=False)
        if not acquired:
            return False
        try:
            if not self.source_dir.exists():
                return False
            for path in self.source_dir.rglob("*"):
                self._copy_file(path)
        except Exception as exc:
            self._sync_errors += 1
            if HF_BRIDGE is not None:
                HF_BRIDGE.event("drive_log_sync_error", {
                    "error": repr(exc), "count": self._sync_errors, "dest_dir": str(self.dest_dir),
                })
        finally:
            self._lock.release()
        return True

    def _loop(self) -> None:
        while not self._stop.wait(self.sync_seconds):
            self.sync_once()

    def start(self) -> None:
        self.sync_once()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self) -> None:
        self._stop.set()
        if self._thread is not None and self._thread is not threading.current_thread():
            self._thread.join(timeout=min(30, self.sync_seconds))
        self.sync_once(wait_for_lock=True)


SENSITIVE_CHILD_ENV_NAMES = {
    "HF_TOKEN", "HF_KEY", "HUGGING_FACE_HUB_TOKEN", "OPENROUTER_API_KEY",
    "KAGGLE_USERNAME", "KAGGLE_KEY", "GH_TOKEN", "GITHUB_TOKEN",
}


def sanitized_child_env(base: dict | None = None, *, allow: set[str] | None = None) -> dict:
    child = dict(os.environ if base is None else base)
    allowed = set(allow or ())
    for name in SENSITIVE_CHILD_ENV_NAMES - allowed:
        child.pop(name, None)
    return child


def run_streamed(
    cmd: list[str],
    *,
    cwd: str | None = None,
    env: dict | None = None,
    check: bool = True,
    label: str | None = None,
) -> subprocess.CompletedProcess:
    label = label or Path(cmd[0]).name
    safe_cmd = [str(x) for x in cmd]
    command_started = time.monotonic()
    if HF_BRIDGE is not None:
        HF_BRIDGE.update_runtime_state(
            active_phase=label,
            active_command=safe_cmd,
            command_started_epoch=time.time(),
            last_progress_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        )
        HF_BRIDGE.event("command_start", {"label": label, "cmd": safe_cmd, "cwd": cwd})
    print(f"[cmd:{label}] {' '.join(safe_cmd)}")
    proc = subprocess.Popen(
        safe_cmd,
        cwd=cwd,
        env=sanitized_child_env() if env is None else env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
    )
    assert proc.stdout is not None
    suppressed_noncritical = 0
    encoding_replacement_count = 0
    for line in proc.stdout:
        encoding_replacement_count += line.count("\ufffd")
        if HF_BRIDGE is not None:
            HF_BRIDGE.update_runtime_state(last_progress_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        if is_noncritical_log_line(line):
            suppressed_noncritical += 1
            continue
        print(line, end="")
    rc = proc.wait()
    if encoding_replacement_count:
        print(f"[cmd:{label}] UTF-8 decoding replacements={encoding_replacement_count}")
        if rc == 0:
            rc = 86
    if suppressed_noncritical:
        print(f"[cmd:{label}] suppressed_noncritical_lines={suppressed_noncritical}")
        if HF_BRIDGE is not None:
            HF_BRIDGE.event(
                "command_suppressed_noncritical_lines",
                {"label": label, "count": suppressed_noncritical},
            )
    print(f"[cmd:{label}] exit={rc}")
    if HF_BRIDGE is not None:
        command_elapsed_s = round(time.monotonic() - command_started, 3)
        HF_BRIDGE.update_runtime_state(
            active_phase="idle" if rc == 0 else "failed",
            active_command=None,
            command_started_epoch=None,
            last_command=label,
            last_command_returncode=rc,
            last_command_elapsed_s=command_elapsed_s,
        )
        HF_BRIDGE.event(
            "command_end",
            {"label": label, "returncode": rc, "duration_s": command_elapsed_s, "stop_reason": "process_exit"},
            upload=(rc != 0),
        )
    if check and rc != 0:
        raise subprocess.CalledProcessError(rc, safe_cmd)
    return subprocess.CompletedProcess(safe_cmd, rc)


section("1. Runtime probe")
try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is not importable in this Colab runtime") from exc

print("python", sys.version)
P141_EXPECTED_PYTHON = (3, 12)
if sys.version_info[:2] != P141_EXPECTED_PYTHON:
    raise RuntimeError(
        f"P141 requires Python {P141_EXPECTED_PYTHON[0]}.{P141_EXPECTED_PYTHON[1]} "
        f"for the validated dependency contract; got {sys.version.split()[0]}"
    )
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime -> Change runtime type -> GPU")

gpu_name = torch.cuda.get_device_name(0)
print("gpu", gpu_name)
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
    capture_output=True,
).stdout.strip())

gpu_count = torch.cuda.device_count()
gpu_capability = torch.cuda.get_device_capability(0)
gpu_bf16_supported = bool(torch.cuda.is_bf16_supported())
if int(FORCE_GPU_COUNT) > gpu_count:
    raise RuntimeError(f"FORCE_GPU_COUNT={FORCE_GPU_COUNT} exceeds visible CUDA devices={gpu_count}")
if TRAIN_PRECISION == "bf16" and not gpu_bf16_supported:
    raise RuntimeError("TRAIN_PRECISION=bf16 is unsupported by this GPU")
RESOLVED_TRAIN_PRECISION = (
    "bf16" if TRAIN_PRECISION == "auto" and gpu_bf16_supported
    else "fp16" if TRAIN_PRECISION == "auto"
    else TRAIN_PRECISION
)
print("precision preflight", json.dumps({
    "gpu_count": gpu_count, "capability": gpu_capability,
    "bf16_supported": gpu_bf16_supported, "resolved": RESOLVED_TRAIN_PRECISION,
}, sort_keys=True))
if "L4" not in gpu_name:
    print("[runtime-note] GPU is not L4; use result as functional evidence, not Kaggle timing proof.")
    if REQUIRE_L4_TIMING:
        raise RuntimeError("REQUIRE_L4_TIMING is enabled, but the selected GPU is not L4")


def secret(name: str) -> str | None:
    try:
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None


section("2. Mount Drive and unpack bundle")
from google.colab import drive, userdata
from importlib import metadata as _bootstrap_metadata

_HF_BRIDGE_VERSION = "0.36.2"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", f"huggingface_hub=={_HF_BRIDGE_VERSION}"],
    check=True,
)
if _bootstrap_metadata.version("huggingface-hub") != _HF_BRIDGE_VERSION:
    raise RuntimeError("HF bridge bootstrap version mismatch")

# Start HF logging before Drive so a DriveFS failure is observable and nonfatal.
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or secret("HF_TOKEN") or secret("HF_KEY") or ""
HF_BRIDGE = HFLogBridge(
    token=os.environ.get("HF_TOKEN", ""),
    run_id=RUN_ID,
    dataset_repo=HF_LOG_DATASET,
    sync_seconds=HF_LOG_SYNC_SECONDS,
)
sys.stdout = TeeStream(sys.__stdout__, HF_BRIDGE.stdout_path)
sys.stderr = TeeStream(sys.__stderr__, HF_BRIDGE.stderr_path)

DRIVE_AVAILABLE = False
DRIVE_MOUNT_ERROR = None


def probe_drive_writable():
    root = Path("/content/drive/MyDrive")
    if not root.is_dir():
        return False, "MyDrive directory is absent"
    probe = root / f".arc2016_drive_probe_{os.getpid()}"
    try:
        probe.write_text("ok", encoding="ascii")
        if probe.read_text(encoding="ascii") != "ok":
            return False, "Drive write probe content mismatch"
        probe.unlink()
        return True, None
    except Exception as exc:
        try:
            probe.unlink(missing_ok=True)
        except Exception:
            pass
        return False, f"{type(exc).__name__}: {exc}"


DRIVE_AVAILABLE, DRIVE_MOUNT_ERROR = probe_drive_writable()
if DRIVE_AVAILABLE:
    print("Drive already mounted and accessible")
elif TRY_DRIVE_MOUNT:
    try:
        drive.mount("/content/drive", timeout_ms=180000)
        DRIVE_AVAILABLE, DRIVE_MOUNT_ERROR = probe_drive_writable()
        if not DRIVE_AVAILABLE:
            print("[runtime-note] Drive mounted but failed writable probe; using HF + /content")
    except Exception as exc:
        DRIVE_MOUNT_ERROR = f"{type(exc).__name__}: {exc}"
        print("[runtime-note] Drive mount unavailable; continuing with embedded bundle and HF logs")
        print("drive mount detail", DRIVE_MOUNT_ERROR)
else:
    DRIVE_MOUNT_ERROR = "disabled_by_TRY_DRIVE_MOUNT"
    print("[runtime-note] Drive mount skipped; using embedded bundle and HF logs")

if DRIVE_AVAILABLE:
    DRIVE_LOG_MIRROR = DriveLogMirror(
        HF_BRIDGE.log_dir,
        Path(DRIVE_LOG_ROOT) / RUN_ID,
        DRIVE_LOG_SYNC_SECONDS,
    )
    DRIVE_LOG_MIRROR.start()
    print("drive live log dir", DRIVE_LOG_MIRROR.dest_dir)
else:
    DRIVE_LOG_MIRROR = None
    print("Drive mirror disabled for this run; HF is the remote evidence store")
if HF_BRIDGE.enabled:
    print("hf log repo", HF_BRIDGE.repo_id)
    print("hf log url", HF_BRIDGE.repo_url)
else:
    print("hf remote logging disabled or unavailable; logs remain local to this runtime")
if not DRIVE_AVAILABLE and not HF_BRIDGE.enabled:
    raise RuntimeError(
        "Drive is unavailable and HF logging could not start. Check HF_TOKEN/HF_KEY "
        "before running a long experiment without a remote evidence store."
    )
print("lab parameters", json.dumps(LAB_PARAMETERS, sort_keys=True))
HF_BRIDGE.event("lab_parameters", LAB_PARAMETERS, upload=HF_BRIDGE.enabled)
HF_BRIDGE.event("drive_mount_status", {
    "available": DRIVE_AVAILABLE,
    "error": DRIVE_MOUNT_ERROR,
    "try_drive_mount": bool(TRY_DRIVE_MOUNT),
}, upload=HF_BRIDGE.enabled)
if DRIVE_LOG_MIRROR is not None:
    HF_BRIDGE.event("drive_log_mirror_started", {
        "dest_dir": str(DRIVE_LOG_MIRROR.dest_dir),
        "sync_seconds": DRIVE_LOG_MIRROR.sync_seconds,
    }, upload=HF_BRIDGE.enabled)


def _colab_excepthook(exc_type, exc, tb):
    finalizer_errors = []
    if HF_BRIDGE is not None:
        try:
            HF_BRIDGE.event("run_failed", {
                "error": repr(exc),
                "traceback": "".join(traceback.format_exception(exc_type, exc, tb))[-12000:],
            }, upload=HF_BRIDGE.enabled)
        except Exception as hook_exc:
            finalizer_errors.append(f"hf_event:{type(hook_exc).__name__}:{hook_exc}")
        try:
            HF_BRIDGE.stop("failed", extra={"error": repr(exc)})
        except Exception as hook_exc:
            finalizer_errors.append(f"hf_stop:{type(hook_exc).__name__}:{hook_exc}")
    if DRIVE_LOG_MIRROR is not None:
        try:
            DRIVE_LOG_MIRROR.stop()
        except Exception as hook_exc:
            finalizer_errors.append(f"drive_stop:{type(hook_exc).__name__}:{hook_exc}")
    if finalizer_errors:
        print("failure finalizer errors", json.dumps(finalizer_errors), file=sys.__stderr__)
    sys.__excepthook__(exc_type, exc, tb)


sys.excepthook = _colab_excepthook

def _ipython_failure_handler(self, etype, value, tb, tb_offset=None):
    try:
        _colab_excepthook(etype, value, tb)
    except Exception as hook_exc:
        print(f"failure finalizer error: {type(hook_exc).__name__}: {hook_exc}", file=sys.__stderr__)
    return traceback.format_exception(etype, value, tb)

try:
    get_ipython().set_custom_exc((BaseException,), _ipython_failure_handler)
except Exception as exc:
    raise RuntimeError(f"Could not install IPython failure finalizer: {exc}") from exc

local_bundle = Path("/content") / BUNDLE_NAME
try:
    embedded_payload = base64.b64decode(EMBEDDED_BUNDLE_B64, validate=True)
except Exception as exc:
    raise RuntimeError(f"Embedded bundle base64 is invalid: {type(exc).__name__}: {exc}") from exc
embedded_sha256 = hashlib.sha256(embedded_payload).hexdigest()
if embedded_sha256 != EMBEDDED_BUNDLE_SHA256:
    raise RuntimeError(
        "Embedded bundle SHA-256 mismatch: "
        f"expected={EMBEDDED_BUNDLE_SHA256} actual={embedded_sha256}"
    )
local_bundle.write_bytes(embedded_payload)
bundle_path = local_bundle
print("bundle source embedded")
print("bundle sha256", embedded_sha256)
print("bundle local path", local_bundle)
print("bundle local size", local_bundle.stat().st_size)

if Path(ROOT_DIR).exists():
    shutil.rmtree(ROOT_DIR)
Path(ROOT_DIR).mkdir(parents=True, exist_ok=True)


def safe_extract_bundle(bundle_zip: zipfile.ZipFile, root: Path) -> None:
    """Extract zips created on Windows or POSIX into a POSIX Colab tree."""
    for member in bundle_zip.infolist():
        raw_name = member.filename
        normalized = raw_name.replace("\\", "/").lstrip("/")
        parts = [part for part in normalized.split("/") if part not in {"", "."}]
        if not parts or any(part == ".." for part in parts):
            raise RuntimeError(f"Unsafe bundle member path: {raw_name!r}")
        target = root.joinpath(*parts)
        if raw_name.endswith(("/", "\\")):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with bundle_zip.open(member) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)


try:
    zip_names = []
    with zipfile.ZipFile(local_bundle) as bundle_zip:
        zip_names = bundle_zip.namelist()
        bad_member = bundle_zip.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt member in bundle zip: {bad_member}")
        print("bundle entries", len(zip_names))
        print("bundle first entries", zip_names[:20])
        print("bundle backslash entries", sum("\\" in name for name in zip_names))
        safe_extract_bundle(bundle_zip, Path(ROOT_DIR))
except zipfile.BadZipFile as exc:
    raise RuntimeError(
        f"Bundle is not a valid zip after Drive copy: {local_bundle} "
        f"({local_bundle.stat().st_size if local_bundle.exists() else 'missing'} bytes)"
    ) from exc

def extracted_tree_sample(root: Path, limit: int = 120) -> list[str]:
    if not root.exists():
        return [f"{root} does not exist"]
    rows = []
    for path in root.rglob("*"):
        try:
            rel = path.relative_to(root).as_posix()
        except ValueError:
            rel = str(path)
        rows.append(rel + ("/" if path.is_dir() else ""))
        if len(rows) >= limit:
            break
    return rows


def resolve_arc2_root(root: Path) -> Path:
    candidates = []
    direct = root / "arc2"
    if direct.exists():
        candidates.append(direct)
    for marker in root.rglob("qwen_worker_throughput.py"):
        if marker.parent.name == "hf":
            candidates.append(marker.parent.parent)
    for candidate in candidates:
        if (
            (candidate / "hf" / "qwen_worker_throughput.py").exists()
            and (candidate / "kaggle_qwen_l4x4" / "qwen_ttt_worker.py").exists()
            and (candidate / "data" / "arc-agi_evaluation_challenges.json").exists()
        ):
            return candidate
    raise RuntimeError(
        "Bundle extracted, but arc2 root was not found. "
        f"root={root} zip_first_entries={zip_names[:40]} "
        f"extracted_tree_sample={extracted_tree_sample(root)}"
    )


ARC2 = resolve_arc2_root(Path(ROOT_DIR))
print("bundle", bundle_path)
print("arc2", ARC2)
print("extracted tree sample", extracted_tree_sample(Path(ROOT_DIR), limit=40))


def verify_bundle_contract(arc2: Path) -> None:
    manifest_path = arc2 / "BUNDLE_MANIFEST.json"
    if not manifest_path.is_file():
        raise RuntimeError("Bundle manifest is missing")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8", errors="strict"))
    files = manifest.get("files") if isinstance(manifest, dict) else None
    if not isinstance(files, dict) or not files:
        raise RuntimeError("Bundle manifest has no files")
    failures = []
    compiled = 0
    for member, expected in sorted(files.items()):
        if not str(member).startswith("arc2/"):
            failures.append(f"{member}:outside_arc2")
            continue
        path = Path(ROOT_DIR) / Path(*PurePosixPath(member).parts)
        if not path.is_file():
            failures.append(f"{member}:missing")
            continue
        raw = path.read_bytes()
        if len(raw) != int(expected.get("size_bytes", -1)):
            failures.append(f"{member}:size")
        if hashlib.sha256(raw).hexdigest() != expected.get("sha256"):
            failures.append(f"{member}:sha256")
        if path.suffix.lower() in {".py", ".json", ".jsonl", ".md", ".txt", ".yaml", ".yml"}:
            try:
                decoded = raw.decode("utf-8", errors="strict")
                if "\ufffd" in decoded:
                    failures.append(f"{member}:replacement_character")
                if path.suffix.lower() == ".py":
                    compile(decoded, str(path), "exec")
                    compiled += 1
            except (UnicodeError, SyntaxError) as exc:
                failures.append(f"{member}:{type(exc).__name__}:{exc}")
    if failures:
        raise RuntimeError(f"Bundle integrity/encoding/compile failures: {failures[:40]}")
    print("bundle contract ok", json.dumps({
        "release": manifest.get("release"),
        "file_count": len(files),
        "compiled_python_files": compiled,
    }, sort_keys=True))

verify_bundle_contract(ARC2)

sys.path.insert(0, str(ARC2 / "kaggle_qwen_l4x4"))
sys.path.insert(0, str(ARC2 / "pipeline"))
from qwen_ttt_worker import parse_run_matrix
from candidate_selector import load_selector_weights, normalize_selector_weights

if RUN_MATRIX:
    RUN_MATRIX = parse_run_matrix(json.dumps(RUN_MATRIX, separators=(",", ":"), sort_keys=True))
    QWEN_OPTIONAL_OVERRIDES["ARC_QWEN_RUN_MATRIX_JSON"] = json.dumps(RUN_MATRIX, separators=(",", ":"), sort_keys=True)
selector_contract = normalize_selector_weights(load_selector_weights(SELECTOR_WEIGHT_SPEC))
for selector_mode in csv_items(SELECTOR_SWEEP_MODES):
    normalize_selector_weights({"selection_mode": selector_mode})
print("early run-matrix/selector contract ok", json.dumps({
    "portfolio_runs": len(RUN_MATRIX),
    "selector_mode": selector_contract["selection_mode"],
    "sweep_modes": csv_items(SELECTOR_SWEEP_MODES),
}, sort_keys=True))


section("3. Kaggle/HF credentials")
for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    os.environ[key] = os.environ.get(key) or secret(key) or ""

drive_kaggle_json = Path("/content/drive/MyDrive/kaggle.json")
if (not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY")) and drive_kaggle_json.exists():
    cfg = json.loads(drive_kaggle_json.read_text(encoding="utf-8"))
    os.environ["KAGGLE_USERNAME"] = cfg.get("username", "")
    os.environ["KAGGLE_KEY"] = cfg.get("key", "")

if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
    raise RuntimeError("Missing Kaggle credentials. Add Colab Secrets or MyDrive/kaggle.json.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / "kaggle.json").write_text(json.dumps({
    "username": os.environ["KAGGLE_USERNAME"],
    "key": os.environ["KAGGLE_KEY"],
}), encoding="utf-8")
os.chmod(kaggle_dir / "kaggle.json", 0o600)
print("kaggle user", os.environ["KAGGLE_USERNAME"])
print("hf token present", bool(os.environ.get("HF_TOKEN")))

if HF_BRIDGE is not None:
    HF_BRIDGE.event("colab_runtime_ready", {
        "lab_config_version": LAB_CONFIG_VERSION,
        "experiment_id": EXPERIMENT_ID,
        "profiles": PROFILES,
        "selector_weight_spec": SELECTOR_WEIGHT_SPEC,
        "gpu": gpu_name,
        "python": sys.version,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "kaggle_user": os.environ["KAGGLE_USERNAME"],
    }, upload=HF_BRIDGE.enabled)


section("4. Atomic dependency install and ABI contract")
from importlib import metadata as importlib_metadata


def pip_check_snapshot():
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True,
        capture_output=True,
        check=False,
    )
    lines = sorted({line.strip() for line in (completed.stdout + "\n" + completed.stderr).splitlines() if line.strip()})
    return {"returncode": completed.returncode, "lines": lines}


def installed_versions(names):
    result = {}
    for name in names:
        try:
            result[name] = importlib_metadata.version(name)
        except importlib_metadata.PackageNotFoundError:
            result[name] = None
    return result


pip_check_pristine = pip_check_snapshot()
tracked_packages = [
    *P141_EXPECTED_EXACT,
    *P141_EXPECTED_RUNTIME_PUBLIC,
    "gradio",
    "gradio-client",
    "hf-gradio",
]
versions_pristine = installed_versions(tracked_packages)
P141_UNUSED_CONFLICT_PACKAGES = ["gradio", "gradio-client", "hf-gradio"]
# Query the removal set directly. It may intentionally contain packages
# outside tracked_packages in future Colab images.
unused_conflicts_before = installed_versions(P141_UNUSED_CONFLICT_PACKAGES)
unused_conflicts_removed = [
    name for name, version in unused_conflicts_before.items() if version is not None
]
if unused_conflicts_removed:
    run_streamed(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y", *unused_conflicts_removed],
        check=True,
        label="pip_remove_unused_conflict_stack",
    )
unused_conflicts_after = installed_versions(P141_UNUSED_CONFLICT_PACKAGES)
pip_check_post_removal = pip_check_snapshot()
versions_post_removal = installed_versions(tracked_packages)
removal_induced_conflicts = sorted(
    set(pip_check_post_removal["lines"]) - set(pip_check_pristine["lines"])
)
if any(unused_conflicts_after.values()) or removal_induced_conflicts:
    raise RuntimeError(
        "P141 conflict-stack removal was not clean: "
        f"remaining={unused_conflicts_after} introduced={removal_induced_conflicts}"
    )

torch_before = torch.__version__
cuda_before = torch.version.cuda
if torch_before.split("+", 1)[0] != P141_EXPECTED_RUNTIME_PUBLIC["torch"]:
    raise RuntimeError(
        f"P141 requires Colab torch public version {P141_EXPECTED_RUNTIME_PUBLIC['torch']}; "
        f"found {torch_before}. Refuse to replace the CUDA runtime in-place."
    )
pip_check_before = pip_check_post_removal
versions_before = versions_post_removal
run_streamed(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "--progress-bar",
        "off",
        *P141_ATOMIC_SPECS,
    ],
    check=True,
    label="pip_p141_atomic_dependency_lock",
)
pip_check_after = pip_check_snapshot()
versions_after = installed_versions([*P141_EXPECTED_EXACT, *P141_EXPECTED_RUNTIME_PUBLIC])
torch_after = importlib_metadata.version("torch")
install_induced_conflicts = sorted(set(pip_check_after["lines"]) - set(pip_check_post_removal["lines"]))
new_pip_conflicts = sorted(set(pip_check_after["lines"]) - set(pip_check_pristine["lines"]))
version_errors = [
    f"{name}: expected {expected}, found {versions_after.get(name)}"
    for name, expected in P141_EXPECTED_EXACT.items()
    if versions_after.get(name) != expected
]
version_errors.extend(
    f"{name}: expected public version {expected}, found {versions_after.get(name)}"
    for name, expected in P141_EXPECTED_RUNTIME_PUBLIC.items()
    if (versions_after.get(name) or "").split("+", 1)[0] != expected
)
if torch.version.cuda != cuda_before:
    version_errors.append(f"CUDA changed unexpectedly: before={cuda_before} after={torch.version.cuda}")
pip_check_execution_errors = [
    f"{name}: returncode={snapshot.get('returncode')!r}"
    for name, snapshot in (
        ("pristine", pip_check_pristine),
        ("post_removal", pip_check_post_removal),
        ("after", pip_check_after),
    )
    if snapshot.get("returncode") not in {0, 1}
]
version_errors.extend(pip_check_execution_errors)

smoke_code = r"""
import torch
import unsloth
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import transformers, peft, datasets, accelerate, trl, bitsandbytes, xformers, sklearn, tokenizers, torchao
from transformers import Qwen3Config, Qwen3ForCausalLM
from peft import LoraConfig, get_peft_model
config = Qwen3Config(
    vocab_size=128, hidden_size=32, intermediate_size=64,
    num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,
    head_dim=8, max_position_embeddings=128,
)
model = Qwen3ForCausalLM(config)
model = get_peft_model(model, LoraConfig(r=2, lora_alpha=4, target_modules=["q_proj"]))
assert any(parameter.requires_grad for parameter in model.parameters())
model = model.cuda()
for parameter in model.parameters():
    if parameter.dtype == torch.float32 and not parameter.requires_grad:
        parameter.data = parameter.data.half()
assert all(parameter.dtype == torch.float32 for parameter in model.parameters() if parameter.requires_grad)
from accelerate import Accelerator
optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=1e-4)
accelerator = Accelerator(mixed_precision="fp16")
model, optimizer = accelerator.prepare(model, optimizer)
tokens = torch.tensor([[1, 2, 3, 4]], device=accelerator.device)
with accelerator.autocast():
    loss = model(input_ids=tokens, labels=tokens).loss
accelerator.backward(loss)
optimizer.step()
optimizer.zero_grad(set_to_none=True)
print("P141_QWEN3_PEFT_LORA_SMOKE_OK")
print("P141_FP16_OPTIMIZER_STEP_SMOKE_OK")
"""
try:
    import_smoke = subprocess.run(
        [sys.executable, "-c", smoke_code],
        text=True,
        capture_output=True,
        check=False,
        timeout=180,
    )
except subprocess.TimeoutExpired as exc:
    import_smoke = subprocess.CompletedProcess(
        exc.cmd,
        124,
        stdout=str(exc.stdout or ""),
        stderr=f"dependency smoke timed out after {exc.timeout}s; partial_stderr={exc.stderr or ''}",
    )
dependency_contract = {
    "schema": "arc-agi-2-colab-dependency-contract-v1",
    "lab_config_version": LAB_CONFIG_VERSION,
    "atomic_specs": P141_ATOMIC_SPECS,
    "unused_conflicts_before": unused_conflicts_before,
    "unused_conflicts_removed": unused_conflicts_removed,
    "unused_conflicts_after": unused_conflicts_after,
    "pip_check_pristine": pip_check_pristine,
    "pip_check_post_removal": pip_check_post_removal,
    "versions_pristine": versions_pristine,
    "versions_post_removal": versions_post_removal,
    "removal_induced_conflicts": removal_induced_conflicts,
    "install_induced_conflicts": install_induced_conflicts,
    "torch_before": torch_before,
    "torch_after": torch_after,
    "cuda_before": cuda_before,
    "cuda_after": torch.version.cuda,
    "versions_before": versions_before,
    "versions_after": versions_after,
    "pip_check_before": pip_check_before,
    "pip_check_after": pip_check_after,
    "new_pip_conflicts": new_pip_conflicts,
    "pip_check_execution_errors": pip_check_execution_errors,
    "version_errors": version_errors,
    "import_smoke": {
        "returncode": import_smoke.returncode,
        "stdout": import_smoke.stdout[-4000:],
        "stderr": import_smoke.stderr[-8000:],
    },
}
torchao_after = installed_versions(["torchao"])["torchao"]
dependency_contract["torchao_after"] = torchao_after
dependency_contract["torchao_imported_version"] = getattr(sys.modules.get("torchao"), "__version__", None)
dependency_contract["loaded_huggingface_hub_version"] = getattr(sys.modules.get("huggingface_hub"), "__version__", None)
if dependency_contract["loaded_huggingface_hub_version"] != P141_EXPECTED_EXACT["huggingface-hub"]:
    version_errors.append("loaded huggingface_hub module version differs from locked distribution")
dependency_contract["version_errors"] = list(version_errors)
DEPENDENCY_CONTRACT_PATH.write_text(
    json.dumps(dependency_contract, indent=2, sort_keys=True),
    encoding="utf-8",
)
print("dependency contract", json.dumps(dependency_contract, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("p141_dependency_contract", dependency_contract, upload=True)
if new_pip_conflicts or version_errors or import_smoke.returncode != 0:
    raise RuntimeError(
        "P141 dependency contract failed before model download: "
        f"new_pip_conflicts={new_pip_conflicts} version_errors={version_errors} "
        f"import_smoke_rc={import_smoke.returncode}"
    )
print("P141 dependency contract passed")


section("5. Stage official Kaggle Qwen model and Unsloth/flash-attn kernel output")
# This downloads roughly 8-9 GB into the Colab runtime. It is reused by all profiles.
env = sanitized_child_env(allow={"KAGGLE_USERNAME", "KAGGLE_KEY"})
env.update({
    "PYTHONUNBUFFERED": "1",
    "PYTHONIOENCODING": "utf-8",
    "PYTHONUTF8": "1",
    "ARC_DOWNLOAD_QWEN_MODEL": "1",
    "ARC_DOWNLOAD_UNSLOTH_KERNEL": "0",
    "ARC_UPGRADE_KAGGLE_CLI": "0",
    "ARC_KAGGLE_OUTPUT_FILE_PATTERN": KAGGLE_OUTPUT_FILE_PATTERN,
    "ARC_KAGGLE_OUTPUT_PAGE_SIZE": "200",
    "ARC_KAGGLE_OUTPUT_MAX_EMPTY_FILTERED_PAGES": "180",
    "ARC_KAGGLE_OUTPUT_MAX_PAGES": "500",
    "ARC_UNSLOTH_DOWNLOAD_FALLBACK_CLI": "1",
    "ARC_PROBE_LOAD_TOKENIZER": "1",
    "ARC_PROBE_IMPORT_PACKAGES": "1",
    "ARC_PROBE_STRICT_FLASH_CAUSAL": FLASH_CAUSAL_STRICT,
})
stage_config = {
    "lab_config_version": LAB_CONFIG_VERSION,
    "experiment_id": EXPERIMENT_ID,
    "file_pattern": env["ARC_KAGGLE_OUTPUT_FILE_PATTERN"],
    "page_size": env["ARC_KAGGLE_OUTPUT_PAGE_SIZE"],
    "max_empty_filtered_pages": env["ARC_KAGGLE_OUTPUT_MAX_EMPTY_FILTERED_PAGES"],
    "max_pages": env["ARC_KAGGLE_OUTPUT_MAX_PAGES"],
    "unsloth_fallback_cli": env["ARC_UNSLOTH_DOWNLOAD_FALLBACK_CLI"],
    "unsloth_download_requested": env["ARC_DOWNLOAD_UNSLOTH_KERNEL"] == "1",
    "staged_dependency_path_mode": "skip",
    "qwen3_overlay_requested": QWEN3_PATCH_OVERLAY,
    "consumed_staged_artifact_count": 0,
    "colab_compat_unsloth_spec": COLAB_COMPAT_UNSLOTH_SPEC,
    "flash_causal_strict": FLASH_CAUSAL_STRICT,
}
if stage_config["unsloth_download_requested"] and stage_config["consumed_staged_artifact_count"] == 0:
    raise RuntimeError("P141 FinOps gate: refusing an Unsloth kernel download with zero consumed artifacts")
print("stage kaggle output config", json.dumps(stage_config, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("stage_kaggle_output_config", stage_config, upload=True)
run_streamed(
    [sys.executable, "-u", str(ARC2 / "hf" / "hf_stage_kaggle_assets.py")],
    cwd=str(ARC2),
    env=env,
    check=True,
    label="stage_kaggle_qwen_unsloth",
)
print("staging complete")


section("5b. Install Colab-compatible Unsloth runtime when needed")
def install_colab_compatible_unsloth() -> dict:
    raw = os.environ.get("ARC_COLAB_INSTALL_COMPAT_UNSLOTH", INSTALL_COMPAT_UNSLOTH).strip().lower()
    if raw in {"1", "true", "yes"}:
        raw = "force"
    if raw in {"0", "false", "no"}:
        raw = "skip"
    if raw not in {"auto", "force", "skip"}:
        raise ValueError(f"Unsupported ARC_COLAB_INSTALL_COMPAT_UNSLOTH={raw!r}")
    enabled = raw == "force" or (raw == "auto" and sys.version_info[:2] != (3, 11))
    report = {
        "requested": raw,
        "enabled": enabled,
        "python": sys.version.split()[0],
        "spec": COLAB_COMPAT_UNSLOTH_SPEC,
    }
    if not enabled:
        print("colab compatible unsloth install skipped", json.dumps(report, sort_keys=True))
        if HF_BRIDGE is not None:
            HF_BRIDGE.event("colab_compatible_unsloth_runtime", report, upload=True)
        return report

    cmd = [
        sys.executable,
        "-c",
        "import unsloth; from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments; print('P141_UNSLOTH_OK')",
    ]
    completed = run_streamed(cmd, check=True, label="verify_p141_colab_unsloth")
    report["verification_returncode"] = completed.returncode
    os.environ["ARC_QWEN_STAGED_DEPENDENCY_PATH_MODE"] = "skip"

    try:
        import importlib.util

        def flash_attn_func_importable() -> bool:
            for module_name in ("flash_attn.flash_attn_interface", "flash_attn"):
                try:
                    module = __import__(module_name, fromlist=["flash_attn_func"])
                    if getattr(module, "flash_attn_func", None) is not None:
                        return True
                except Exception:
                    continue
            return False

        staged_qwen3 = Path("/tmp/pip-install-unsloth-flash-patch/unsloth/models/qwen3.py")
        spec = importlib.util.find_spec("unsloth")
        if spec is None or spec.origin is None:
            raise RuntimeError("pip-installed unsloth package not found after install")
        unsloth_pkg = Path(spec.origin).resolve().parent
        target_qwen3 = unsloth_pkg / "models" / "qwen3.py"
        staged_text = staged_qwen3.read_text(encoding="utf-8", errors="strict") if staged_qwen3.exists() else ""
        staged_uses_flash_attn = "flash_attn_func(" in staged_text
        flash_attn_ready = flash_attn_func_importable()
        overlay_requested = QWEN3_PATCH_OVERLAY in {"1", "true", "yes", "force"}
        overlay_report = {
            "requested": QWEN3_PATCH_OVERLAY,
            "overlay_requested": overlay_requested,
            "staged": str(staged_qwen3),
            "staged_exists": staged_qwen3.exists(),
            "staged_uses_flash_attn_func": staged_uses_flash_attn,
            "flash_attn_func_importable": flash_attn_ready,
            "target": str(target_qwen3),
            "target_exists": target_qwen3.exists(),
        }
        if not overlay_requested:
            overlay_report.update({
                "skipped": True,
                "reason": "disabled_by_default_to_avoid_colab_flash_attn_nameerror",
            })
        elif not staged_qwen3.exists() or not target_qwen3.exists():
            overlay_report.update({
                "skipped": True,
                "reason": "staged_or_target_qwen3_missing",
            })
        elif staged_uses_flash_attn and not flash_attn_ready and QWEN3_PATCH_OVERLAY != "force":
            overlay_report.update({
                "skipped": True,
                "reason": "staged_qwen3_requires_flash_attn_func_but_runtime_cannot_import_it",
            })
        else:
            backup = target_qwen3.with_suffix(".py.before_arc_stage_patch")
            if not backup.exists():
                shutil.copy2(target_qwen3, backup)
            shutil.copy2(staged_qwen3, target_qwen3)
            overlay_report.update({
                "skipped": False,
                "backup": str(backup),
                "bytes": target_qwen3.stat().st_size,
            })
        report["qwen3_patch_overlay"] = overlay_report
    except Exception as exc:
        report["qwen3_patch_overlay_error"] = f"{type(exc).__name__}: {str(exc)[:300]}"
        raise

    print("colab compatible unsloth runtime", json.dumps(report, sort_keys=True))
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("colab_compatible_unsloth_runtime", report, upload=True)
    return report


COLAB_COMPAT_UNSLOTH_REPORT = install_colab_compatible_unsloth()



section("5b. Build leakage-safe public training LOPO episodes")
PILOT_TASKS = int(PILOT_TASKS)
EPISODES_PER_TASK = int(EPISODES_PER_TASK)
OUTER_FOLDS = int(OUTER_FOLDS)
AUTOLEARN_SEED = int(AUTOLEARN_SEED)
BOOTSTRAP_SAMPLES = int(BOOTSTRAP_SAMPLES)
if not OUTER_FOLDS <= PILOT_TASKS <= 16:
    raise ValueError("PILOT_TASKS must be between OUTER_FOLDS and 16 for P141")
if not 1 <= EPISODES_PER_TASK <= 3:
    raise ValueError("EPISODES_PER_TASK must be between 1 and 3")
if not 2 <= OUTER_FOLDS <= 5:
    raise ValueError("OUTER_FOLDS must be between 2 and 5")
if not 100 <= BOOTSTRAP_SAMPLES <= 10000:
    raise ValueError("BOOTSTRAP_SAMPLES must be between 100 and 10000")

AUTOLEARN_RUN_ID = f"p141_{RUN_ID}"
AUTOLEARN_ROOT = Path("/content/arc2_autolearning_runs") / AUTOLEARN_RUN_ID
EPISODE_DIR = AUTOLEARN_ROOT / "episodes"
RANKER_DIR = AUTOLEARN_ROOT / "ranker"
PROCESS_TRACE_PATH = AUTOLEARN_ROOT / "qwen_process_trace.jsonl"
AUTOLEARN_ROOT.mkdir(parents=True, exist_ok=True)
episode_command = [
    sys.executable, "-u", str(ARC2 / "pipeline" / "autolearning_episode_builder.py"),
    "--challenges", str(ARC2 / "data" / "arc-agi_training_challenges.json"),
    "--solutions", str(ARC2 / "data" / "arc-agi_training_solutions.json"),
    "--out-dir", str(EPISODE_DIR),
    "--task-limit", str(PILOT_TASKS),
    "--episodes-per-task", str(EPISODES_PER_TASK),
    "--folds", str(OUTER_FOLDS),
    "--seed", str(AUTOLEARN_SEED),
]
run_streamed(episode_command, cwd=str(ARC2), env=sanitized_child_env(), check=True, label="build_lopo_episodes")
LOPO_CHALLENGES = EPISODE_DIR / "lopo_challenges.json"
LOPO_SOLUTIONS = EPISODE_DIR / "lopo_solutions.json"
EPISODE_MANIFEST_PATH = EPISODE_DIR / "episode_manifest.json"
EPISODE_MANIFEST = json.loads(EPISODE_MANIFEST_PATH.read_text(encoding="utf-8"))
EFFECTIVE_LOPO_KEYS_LIST = tuple(sorted(row["episode_id"] for row in EPISODE_MANIFEST["records"]))
EFFECTIVE_LOPO_KEYS = ",".join(EFFECTIVE_LOPO_KEYS_LIST)
RUN_KEYS_LIST = list(EFFECTIVE_LOPO_KEYS_LIST)
RUN_KEYS = EFFECTIVE_LOPO_KEYS
MAX_TASKS = len(EFFECTIVE_LOPO_KEYS_LIST)
LAB_PARAMETERS["effective_lopo_keys"] = EFFECTIVE_LOPO_KEYS
LAB_PARAMETERS["effective_lopo_key_count"] = MAX_TASKS
if MAX_TASKS != PILOT_TASKS * EPISODES_PER_TASK:
    raise RuntimeError(f"episode count mismatch: {MAX_TASKS}")
if any("output" in test for task in json.loads(LOPO_CHALLENGES.read_text(encoding="utf-8")).values() for test in task["test"]):
    raise RuntimeError("label leakage: held-out output found in LOPO challenges")
lopo_summary = {
    "run_id": AUTOLEARN_RUN_ID,
    "task_count": PILOT_TASKS,
    "episode_count": MAX_TASKS,
    "configured_run_keys": CONFIGURED_RUN_KEYS,
    "effective_lopo_keys": EFFECTIVE_LOPO_KEYS,
    "configured_keys_are_selection_input": False,
    "fold_episode_counts": EPISODE_MANIFEST["fold_episode_counts"],
    "challenge_sha256": EPISODE_MANIFEST["episode_challenges_sha256"],
    "solution_sha256": EPISODE_MANIFEST["episode_solutions_sha256"],
    "label_boundary": EPISODE_MANIFEST["label_boundary"],
}
print("LOPO", json.dumps(lopo_summary, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("lopo_episodes_ready", lopo_summary, upload=True)

section("6. Run frozen Qwen control over LOPO episodes")
def run_profile(profile: str) -> dict:
    run_id = f"colab_{profile}_{int(time.time())}"
    profile_env = sanitized_child_env()
    profile_env.update({
        "PYTHONUNBUFFERED": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTHONUTF8": "1",
        "ARC_QWEN_PROFILE": profile,
        "ARC_QWEN_RUN_TAG": f"colab-{profile}",
        "ARC_QWEN_GPUS": FORCE_GPU_COUNT,
        "ARC_QWEN_THROUGHPUT_GPUS": FORCE_GPU_COUNT,
        "ARC_QWEN_THROUGHPUT_RUN_ID": run_id,
        "ARC_QWEN_THROUGHPUT_KEYS": RUN_KEYS,
        "ARC_QWEN_THROUGHPUT_CHALLENGES": str(LOPO_CHALLENGES),
        "ARC_QWEN_THROUGHPUT_SOLUTIONS": str(LOPO_SOLUTIONS),
        "ARC_QWEN_PROCESS_TRACE": str(PROCESS_TRACE_PATH),
        "ARC_QWEN_TRACE_BATCHES": "1" if ENABLE_FULL_PROCESS_TRACE else "0",
        "ARC_QWEN_TRACE_FILES": "1" if ENABLE_FULL_PROCESS_TRACE else "0",
        "ARC_AUTOLEARN_RUN_ID": AUTOLEARN_RUN_ID,
        "ARC_AUTOLEARN_SOLUTIONS": str(LOPO_SOLUTIONS),
        "ARC_AUTOLEARN_LABEL_BOUNDARY": "post_generation_only",
        "ARC_QWEN_THROUGHPUT_MAX_TASKS": str(MAX_TASKS),
        "ARC_QWEN_THROUGHPUT_SECONDS": str(SECONDS_PER_PROFILE),
        "ARC_QWEN_TASK_ORDER": "complexity_desc",
        "ARC_QWEN_MODEL_DIR": "/tmp/qwen3_4b_grids15_sft139",
        "ARC_SELECTOR_WEIGHTS": SELECTOR_WEIGHT_SPEC,
        "ARC_QWEN_SELECTOR_SWEEP": "1" if SELECTOR_SWEEP_ENABLED else "0",
        "ARC_QWEN_SELECTOR_SWEEP_MODES": SELECTOR_SWEEP_MODES,
        "ARC_MAX_DUPLICATE_ATTEMPT_RATE": str(MAX_DUPLICATE_ATTEMPT_RATE),
        "ARC_QWEN_THROUGHPUT_REQUIRE_PROBE": "1",
        "ARC_QWEN_THROUGHPUT_FAIL_ON_INVALID_CANDIDATE_FILES": "1",
        "ARC_QWEN_THROUGHPUT_USE_SYMBOLIC": "1" if USE_SYMBOLIC else "0",
        "ARC_MISSING_SYMBOLIC_FALLBACK": "1" if MISSING_SYMBOLIC_FALLBACK else "0",
        "ARC_PROBE_LOAD_TOKENIZER": "1",
        "ARC_PROBE_IMPORT_PACKAGES": "1",
        "ARC_PROBE_RECURSIVE_SEARCH": "1",
        "ARC_PROBE_STRICT_FLASH_CAUSAL": FLASH_CAUSAL_STRICT,
        "ARC_EXPERIMENT_ID": EXPERIMENT_ID,
        "ARC_EXPERIMENT_NOTE": EXPERIMENT_NOTE,
    })
    profile_env.update(QWEN_OPTIONAL_OVERRIDES)
    if os.environ.get("ARC_QWEN_STAGED_DEPENDENCY_PATH_MODE"):
        profile_env["ARC_QWEN_STAGED_DEPENDENCY_PATH_MODE"] = os.environ["ARC_QWEN_STAGED_DEPENDENCY_PATH_MODE"]
    completed = run_streamed(
        [sys.executable, "-u", str(ARC2 / "hf" / "qwen_worker_throughput.py")],
        cwd=str(ARC2),
        env=profile_env,
        check=False,
        label=f"qwen_throughput_{profile}",
    )
    rc = completed.returncode
    report_path = Path("/tmp/arc_qwen_throughput") / run_id / "qwen_throughput_report.json"
    if not report_path.exists():
        raise FileNotFoundError(report_path)
    report = json.loads(report_path.read_text(encoding="utf-8"))
    report["process_returncode"] = rc
    report["report_path"] = str(report_path)
    return report


def _exact_zero_returncode(report: dict, key: str) -> bool:
    value = report.get(key)
    return isinstance(value, int) and not isinstance(value, bool) and value == 0


def profile_is_clean(report: dict) -> bool:
    coverage = report.get("coverage") if isinstance(report.get("coverage"), dict) else {}
    expected = report.get("expected_outputs")
    covered = coverage.get("outputs_with_qwen_candidates", coverage.get("covered_outputs"))
    return (
        report.get("status") == "ok"
        and all(_exact_zero_returncode(report, key) for key in (
            "process_returncode", "probe_returncode", "worker_returncode", "postprocess_returncode"
        ))
        and bool(report.get("format_ok")) is True
        and bool(report.get("portfolio_complete")) is True
        and bool(report.get("selector_sweep_ok")) is True
        and bool(report.get("portfolio_analysis_ok")) is True
        and isinstance(expected, int) and not isinstance(expected, bool)
        and expected == MAX_TASKS and expected > 0
        and isinstance(covered, int) and not isinstance(covered, bool)
        and covered == expected
    )


reports = []
for profile in PROFILES:
    section(f"RUN PROFILE {profile}")
    report = run_profile(profile)
    reports.append(report)
    if profile == PROFILES[0] and STOP_AFTER_BASELINE_FAILURE and not profile_is_clean(report):
        payload = {
            "profile": profile,
            "status": report.get("status"),
            "process_returncode": report.get("process_returncode"),
            "probe_returncode": report.get("probe_returncode"),
            "worker_returncode": report.get("worker_returncode"),
            "postprocess_returncode": report.get("postprocess_returncode"),
            "format_ok": report.get("format_ok"),
            "coverage": report.get("coverage"),
            "candidate_count": report.get("candidate_count"),
            "selector_score": report.get("selector_score"),
            "oracle_score": report.get("oracle_score"),
            "reason": "baseline_not_clean_stop_requested",
        }
        print("baseline failed clean gate; stopping remaining profiles", json.dumps(payload, sort_keys=True))
        if HF_BRIDGE is not None:
            HF_BRIDGE.event("baseline_clean_gate_failed_stop_remaining_profiles", payload, upload=True)
        break


section("7. Summarize, gate, and persist reports to Drive")
sys.path.insert(0, str(ARC2 / "pipeline"))
from generation_candidate_decision import decide_generation_candidate


def pick(report: dict, *path: str, default=None):
    cur = report
    for key in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
    return default if cur is None else cur


rows = []
for report in reports:
    rows.append({
        "profile": report.get("profile"),
        "status": report.get("status"),
        "rc": report.get("process_returncode"),
        "worker_elapsed_s": report.get("worker_elapsed_s"),
        "candidate_count": report.get("candidate_count"),
        "covered": pick(report, "coverage", "outputs_with_qwen_candidates"),
        "expected": report.get("expected_outputs"),
        "unique_candidate_grids_median": pick(report, "candidate_diversity", "unique_candidate_grids_median"),
        "one_unique_candidate_outputs": pick(report, "candidate_diversity", "one_unique_candidate_outputs"),
        "attempt2_input_fallback_outputs": pick(report, "candidate_diversity", "attempt2_input_fallback_outputs"),
        "selector_score": report.get("selector_score"),
        "oracle_score": report.get("oracle_score"),
        "recoverable_selector_gap": report.get("recoverable_selector_gap"),
        "selector_sweep_best": pick(report, "selector_sweep_best", "name"),
        "selector_sweep_best_score": pick(report, "selector_sweep_best", "selector_score"),
        "portfolio_status": pick(report, "portfolio", "status"),
        "portfolio_completed_runs": pick(report, "portfolio", "summary", "completed_runs"),
        "portfolio_oracle_overlap": pick(report, "portfolio_seed_analysis", "oracle_overlap"),
        "portfolio_order_sensitivity": pick(report, "portfolio_seed_analysis", "order_sensitivity"),
        "estimated_hours_for_259_outputs": report.get("estimated_hours_for_259_outputs"),
        "report_path": report.get("report_path"),
    })
print(json.dumps(rows, indent=2))

decision_json = {
    "status": "not_applicable",
    "action": "use_generation_candidate_decision",
    "reason": "P141 has one outer profile; seed-a and seed-b are internal portfolio runs",
    "official_kaggle_score_claim": None,
}
print("\nOUTER A/B DECISION")
print(json.dumps(decision_json, indent=2))

results_root = (
    Path("/content/drive/MyDrive/arc2016_colab_results")
    if DRIVE_AVAILABLE
    else Path("/content/arc2016_colab_results")
)
out_dir = results_root / RUN_ID
out_dir.mkdir(parents=True, exist_ok=True)
summary_path = out_dir / "summary.json"
decision_path = out_dir / "qwen_ab_decision.json"
lab_parameters_path = out_dir / "lab_parameters.json"
summary_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
decision_path.write_text(json.dumps(decision_json, indent=2), encoding="utf-8")
lab_parameters_path.write_text(json.dumps(LAB_PARAMETERS, indent=2, sort_keys=True), encoding="utf-8")
artifact_paths = [summary_path, decision_path, lab_parameters_path, DEPENDENCY_CONTRACT_PATH]
for report in reports:
    profile = str(report.get("profile") or "profile")
    artifact_map = report.get("artifacts") if isinstance(report.get("artifacts"), dict) else {}
    sources = {
        "throughput_report": report.get("report_path"),
        "manifest": artifact_map.get("manifest"),
        "preflight": artifact_map.get("preflight"),
        "candidate_eval": artifact_map.get("candidate_eval"),
        "diagnostics": artifact_map.get("diagnostics"),
        "submission": artifact_map.get("submission"),
        "selector_sweep": artifact_map.get("selector_sweep"),
        "portfolio_report": artifact_map.get("portfolio_report"),
        "portfolio_seed_analysis": artifact_map.get("portfolio_seed_analysis"),
    }
    for label, src_value in sources.items():
        if not src_value:
            continue
        src = Path(src_value)
        if not src.exists() or not src.is_file():
            continue
        suffix = src.suffix or ".json"
        dst = out_dir / f"{profile}_{label}{suffix}"
        dst.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")
        artifact_paths.append(dst)

GENERATION_EVIDENCE_SCOPE = "public_training_lopo_proxy_not_kaggle_score"
generation_decision = decide_generation_candidate(
    reports[0].get("portfolio_seed_analysis") if reports else None,
    reports[0] if reports else None,
    control_tag="seed-a",
    candidate_tag="seed-b",
    min_outputs=FROZEN_P137_REFERENCE["outputs_total"],
    max_attempt2_input_fallback_rate=MAX_ATTEMPT2_INPUT_FALLBACK_RATE,
    reference_control=FROZEN_P137_REFERENCE,
)
generation_decision_json = generation_decision.to_dict()
generation_decision_path = out_dir / "generation_candidate_decision.json"
generation_decision_path.write_text(json.dumps(generation_decision_json, indent=2, sort_keys=True), encoding="utf-8")
artifact_paths.append(generation_decision_path)
print("GENERATION CANDIDATE DECISION")
print(json.dumps(generation_decision_json, indent=2, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("generation_candidate_decision", generation_decision_json, upload=True)

print("saved", out_dir)


section("8. Post-generation label join, cross-fit predictor, and selector replay")
if not reports:
    raise RuntimeError("no Qwen control report exists")
control_report = reports[0]
control_artifacts = control_report.get("artifacts") if isinstance(control_report.get("artifacts"), dict) else {}
if not profile_is_clean(control_report):
    failure_payload = {
        "status": "autolearning_skipped",
        "reason": "qwen_control_not_clean",
        "worker_returncode": control_report.get("worker_returncode"),
        "postprocess_returncode": control_report.get("postprocess_returncode"),
        "format_ok": control_report.get("format_ok"),
        "coverage": control_report.get("coverage"),
        "candidate_count": control_report.get("candidate_count"),
        "rule": "never train or evaluate a ranker on incomplete generator evidence",
    }
    failure_path = AUTOLEARN_ROOT / "autolearning_skipped_unclean_control.json"
    failure_path.write_text(json.dumps(failure_payload, indent=2, sort_keys=True), encoding="utf-8")
    failure_output_dir = out_dir / "autolearning_failure"
    failure_output_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(failure_path, failure_output_dir / failure_path.name)
    if PROCESS_TRACE_PATH.is_file():
        shutil.copy2(PROCESS_TRACE_PATH, failure_output_dir / PROCESS_TRACE_PATH.name)
    failure_archive = Path(shutil.make_archive(
        str(AUTOLEARN_ROOT.parent / f"{AUTOLEARN_RUN_ID}_failed_artifacts"),
        "zip",
        root_dir=AUTOLEARN_ROOT,
    ))
    failure_drive_archive = failure_output_dir / failure_archive.name
    shutil.copy2(failure_archive, failure_drive_archive)
    artifact_paths.extend([failure_path, failure_archive, failure_drive_archive])
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("autolearning_skipped_unclean_control", failure_payload, upload=True)
        HF_BRIDGE.sync_once(extra_paths=artifact_paths)
    if DRIVE_LOG_MIRROR is not None:
        DRIVE_LOG_MIRROR.sync_once()
    raise RuntimeError(
        "P141 stopped before autolearning because the Qwen control was not clean; "
        "diagnostics were sealed before exit"
    )
CANDIDATE_MANIFEST_PATH = Path(str(control_artifacts.get("manifest") or ""))
if not CANDIDATE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"candidate manifest missing: {CANDIDATE_MANIFEST_PATH}")

try:
    import sklearn
    sklearn_version = sklearn.__version__
except ImportError as exc:
    raise RuntimeError("scikit-learn disappeared after the P141 dependency contract") from exc

ranker_command = [
    sys.executable, "-u", str(ARC2 / "pipeline" / "autolearning_ranker.py"),
    "--challenges", str(LOPO_CHALLENGES),
    "--solutions", str(LOPO_SOLUTIONS),
    "--episode-manifest", str(EPISODE_MANIFEST_PATH),
    "--candidates", str(CANDIDATE_MANIFEST_PATH),
    "--process-trace", str(PROCESS_TRACE_PATH),
    "--out-dir", str(RANKER_DIR),
    "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
    "--seed", str(AUTOLEARN_SEED),
    "--min-comparable-episodes", str(MAX_TASKS),
]
autolearning_failure_should_raise = False
ranker_process = run_streamed(
    ranker_command, cwd=str(ARC2 / "pipeline"), env=sanitized_child_env(), check=False, label="crossfit_autolearning_ranker"
)
if ranker_process.returncode != 0:
    failure_path = AUTOLEARN_ROOT / "autolearning_failure.json"
    failure_path.write_text(json.dumps({
        "status": "failed",
        "returncode": ranker_process.returncode,
        "candidate_manifest": str(CANDIDATE_MANIFEST_PATH),
        "process_trace": str(PROCESS_TRACE_PATH),
        "rule": "fail closed; no learned selector is promoted",
    }, indent=2), encoding="utf-8")
    artifact_paths.append(failure_path)
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("autolearning_failed", json.loads(failure_path.read_text(encoding="utf-8")), upload=True)
    autolearning_failure_should_raise = bool(STOP_ON_AUTOLEARN_FAILURE)
else:
    autolearning_report_path = RANKER_DIR / "autolearning_report.json"
    AUTOLEARNING_REPORT = json.loads(autolearning_report_path.read_text(encoding="utf-8"))
    AUTOLEARNING_REPORT["sklearn_version"] = sklearn_version
    AUTOLEARNING_REPORT["qwen_control_status"] = control_report.get("status")
    AUTOLEARNING_REPORT["official_kaggle_score_claim"] = None
    autolearning_report_path.write_text(json.dumps(AUTOLEARNING_REPORT, indent=2, sort_keys=True), encoding="utf-8")
    print("AUTOLEARNING REPORT")
    print(json.dumps(AUTOLEARNING_REPORT, indent=2, sort_keys=True))
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("autolearning_complete", AUTOLEARNING_REPORT, upload=True)

control_work = Path(str(control_report.get("work") or ""))
if control_work.is_dir():
    shutil.copytree(control_work, AUTOLEARN_ROOT / "qwen_control_workdir", dirs_exist_ok=True)
else:
    raise FileNotFoundError(f"Qwen control workdir missing: {control_work}")
archive_base = AUTOLEARN_ROOT.parent / f"{AUTOLEARN_RUN_ID}_complete_artifacts"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=AUTOLEARN_ROOT))
autolearn_output_dir = out_dir / "autolearning"
autolearn_output_dir.mkdir(parents=True, exist_ok=True)
drive_archive = autolearn_output_dir / archive_path.name
shutil.copy2(archive_path, drive_archive)
key_autolearn_paths = [
    EPISODE_MANIFEST_PATH,
    RANKER_DIR / "autolearning_report.json",
    RANKER_DIR / "metric_trace.jsonl",
    RANKER_DIR / "task_trace.jsonl",
    RANKER_DIR / "selection_trace.jsonl",
    PROCESS_TRACE_PATH,
]
for source_path in key_autolearn_paths:
    if source_path.is_file():
        shutil.copy2(source_path, autolearn_output_dir / source_path.name)
artifact_paths.extend([archive_path, drive_archive, *[path for path in key_autolearn_paths if path.is_file()]])
print("autolearning archive", archive_path, archive_path.stat().st_size)
if autolearning_failure_should_raise:
    if HF_BRIDGE is not None:
        HF_BRIDGE.sync_once(extra_paths=artifact_paths)
    if DRIVE_LOG_MIRROR is not None:
        DRIVE_LOG_MIRROR.sync_once()
    raise RuntimeError(f"autolearning ranker failed rc={ranker_process.returncode}; diagnostics were sealed")

if HF_BRIDGE is not None:
    HF_BRIDGE.stop(
        "complete",
        extra_paths=artifact_paths,
        extra={
            "output_dir": str(out_dir),
            "drive_available": DRIVE_AVAILABLE,
            "decision": decision_json,
            "generation_candidate_decision": generation_decision_json,
            "rows": rows,
            "lab_parameters": LAB_PARAMETERS,
        },
    )
if DRIVE_LOG_MIRROR is not None:
    DRIVE_LOG_MIRROR.stop()
print("\nDONE. P141 completed TorchAO-compatible LOPO evidence; it did not submit to Kaggle.")